# N0 — War Room Brief

## CFO Liquidity War Room

It is 9:00 AM. The CFO wants a defensible 30-day liquidity decision before the
end of the session. Your team must decide which assumptions to trust, which
actions to authorize, how much funding and FX protection to use, and what
evidence should trigger escalation.

### Your role

Work as a treasury decision team, not as notebook operators. Every setting below
must be defended in the final CFO review.


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.0.0-alpha.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N0')


## Decision charter

Edit the values below. Stay within the policy and facility limits shown after
the cell. Different teams may use different scenario variants and reach
different defensible answers.


In [ ]:
DECISIONS = {
    'team_name': 'Team Delta',
    'scenario_variant': 'base',  # base, customer_shock, supplier_shock, fx_shock
    'forecast_view': 'p75',      # expected or p75
    'collection_strategy': 'targeted',  # none, targeted, broad
    'payables_extension_days': 5,
    'inventory_release_pct': 0.05,
    'facility_draw': 4_000_000,
    'proposed_hedge_ratios': {'EUR': 0.65, 'GBP': 0.60, 'JPY': 0.60, 'INR': 0.55},
}

print('Team decision charter')
display(pd.DataFrame([
    ('Minimum liquidity', manifest['minimum_liquidity']),
    ('Facility capacity', manifest['credit_facility']['committed_capacity']),
    ('CFO draw threshold', manifest['credit_facility']['cfo_approval_threshold']),
    ('FX policy minimum', manifest['fx_policy']['minimum_hedge_ratio']),
    ('FX policy maximum', manifest['fx_policy']['maximum_hedge_ratio']),
], columns=['constraint', 'value']))
display(pd.Series(DECISIONS, name='team choice').to_frame())
viz.decision_posture(DECISIONS, manifest, OUTPUT_DIR)


## Commit the charter

Running this cell validates the choices and creates a traceable starting point
for the remaining modules. You may revise the charter later, but document why.


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']}")


In [ ]:
charter = f'''# TEAM DECISION CHARTER

**Team:** {DECISIONS['team_name']}  
**Scenario:** {manifest['scenario_variants'][DECISIONS['scenario_variant']]['label']}  
**Forecast view:** {DECISIONS['forecast_view'].upper()}

## Initial position

- Collections: {DECISIONS['collection_strategy']}
- Supplier extension: up to {DECISIONS['payables_extension_days']} days
- Inventory release: {DECISIONS['inventory_release_pct']:.1%}
- Facility draw: ${DECISIONS['facility_draw']:,.0f}

## Evidence required before final approval

1. Data passes all blocking integrity checks.
2. The payment model beats a simple benchmark on a chronological holdout.
3. The selected action forecast protects minimum liquidity.
4. Funding and hedge choices stay within authority or name the exception.
'''
(OUTPUT_DIR / 'N0_team_decision_charter.md').write_text(charter, encoding='utf-8')
display(Markdown(charter))


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
